In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#Import Dataset
!gdown 18h-WsozeS8-nnd6Kg86niuJyB0GQkKNU

Downloading...
From: https://drive.google.com/uc?id=18h-WsozeS8-nnd6Kg86niuJyB0GQkKNU
To: /content/insurance.csv
100% 30.8k/30.8k [00:00<00:00, 39.7MB/s]


In [ ]:
df = pd.read_csv('insurance.csv')

In [ ]:
df

,Age,Diabetes,BloodPressureProblems,AnyTransplants,AnyChronicDiseases,Height,Weight,KnownAllergies,HistoryOfCancerInFamily,NumberOfMajorSurgeries,PremiumPrice
0,45,0,0,0,0,155,57,0,0,0,25000
1,60,1,0,0,0,180,73,0,0,0,29000
2,36,1,1,0,0,158,59,0,0,1,23000
3,52,1,1,0,1,183,93,0,0,2,28000
4,38,0,0,0,1,166,88,0,0,1,23000
...,...,...,...,...,...,...,...,...,...,...,...
981,18,0,0,0,0,169,67,0,0,0,15000
982,64,1,1,0,0,153,70,0,0,3,28000
983,56,0,1,0,0,155,71,0,0,1,29000
984,47,1,1,0,0,158,73,1,0,1,39000


##Feature Engineering

In [ ]:
#Create BMI index

df['BMI'] = round(df['Weight'] / (df['Height']/100)**2,2)

In [ ]:
#Create Total Risk Index

df['TotalRisk'] = df['Diabetes'] + df['BloodPressureProblems'] + df['AnyTransplants'] + df['AnyChronicDiseases'] + df['KnownAllergies'] + df['HistoryOfCancerInFamily'] + df['NumberOfMajorSurgeries']

In [ ]:
df.head()

,Age,Diabetes,BloodPressureProblems,AnyTransplants,AnyChronicDiseases,Height,Weight,KnownAllergies,HistoryOfCancerInFamily,NumberOfMajorSurgeries,PremiumPrice,BMI,TotalRisk
0,45,0,0,0,0,155,57,0,0,0,25000,23.73,0
1,60,1,0,0,0,180,73,0,0,0,29000,22.53,1
2,36,1,1,0,0,158,59,0,0,1,23000,23.63,3
3,52,1,1,0,1,183,93,0,0,2,28000,27.77,5
4,38,0,0,0,1,166,88,0,0,1,23000,31.93,2


#Random Forest

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Define your features (x_v2) and target (y)
# Ensure these column names match your actual dataframe exactly!
x_v2 = df[['Age', 'AnyTransplants', 'AnyChronicDiseases', 'HistoryOfCancerInFamily', 'NumberOfMajorSurgeries', 'BMI', 'BloodPressureProblems','Diabetes', 'KnownAllergies']]
y = df['PremiumPrice']

# 2. Split the data (The missing step)
x_train, x_test, y_train, y_test = train_test_split(x_v2, y, test_size=0.2, random_state=42)

# 3. Scale the data (Creating the variables your error mentioned)
scaler = StandardScaler()
x_train_scaled_v2 = scaler.fit_transform(x_train)
x_test_scaled_v2 = scaler.transform(x_test)

# --- Now the Random Forest will work ---

# Initialize and Train
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(x_train_scaled_v2, y_train)

# Predict and Evaluate
y_pred_rf = rf_model.predict(x_test_scaled_v2)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"--- Random Forest Results ---")
print(f"R2 Score: {r2_rf:.4f}")

# Feature Importance
rf_importance = pd.DataFrame({'Feature': x_v2.columns, 'Importance': rf_model.feature_importances_})
print("\nFeature Importance (Random Forest):")
print(rf_importance.sort_values(by='Importance', ascending=False))

mae_rf = mean_absolute_error(y_test, y_pred_rf)
print(f"Random Forest MAE: ₹{mae_rf:.2f}")

--- Random Forest Results ---
R2 Score: 0.8654

Feature Importance (Random Forest):
                   Feature  Importance
0                      Age    0.637714
5                      BMI    0.147683
1           AnyTransplants    0.096713
2       AnyChronicDiseases    0.039411
4   NumberOfMajorSurgeries    0.030881
3  HistoryOfCancerInFamily    0.022756
6    BloodPressureProblems    0.010380
7                 Diabetes    0.008545
8           KnownAllergies    0.005916
Random Forest MAE: ₹1419.01


In [ ]:

import pickle

# Save the Random Forest model
with open('insurance_rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# Save the Scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Files saved successfully!")

Files saved successfully!


In [ ]:
!pip install -q streamlit pyngrok
from pyngrok import ngrok

# PASTE YOUR TOKEN BELOW
NGROK_AUTH_TOKEN = "3ClD2YC58oF0TVMOlHeihhIxDg5_5CrVG4bt7UAhyRbCqaLCe"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
%%writefile app.py
import streamlit as st
import pickle
import numpy as np
import pandas as pd

# Load assets
model = pickle.load(open('insurance_rf_model.pkl', 'rb'))
scaler = pickle.load(open('scaler.pkl', 'rb'))

st.title("🛡️ Insurance Premium Predictor")

col1, col2 = st.columns(2)

with col1:
    age = st.slider("Age", 18, 66, 30)
    bmi = st.number_input("BMI", 10.0, 60.0, 25.0)
    surgeries = st.selectbox("Major Surgeries", [0, 1, 2, 3])
    bp_problems = st.radio("Blood Pressure Problems?", ["No", "Yes"])

with col2:
    transplant = st.radio("History of Transplants?", ["No", "Yes"])
    chronic = st.radio("Any Chronic Diseases?", ["No", "Yes"])
    cancer = st.radio("Family Cancer History?", ["No", "Yes"])
    diabetes = st.radio("Diabetes?", ["No", "Yes"])
    allergies = st.radio("Known Allergies?", ["No", "Yes"])

if st.button("Calculate Premium"):
    # Convert inputs to 0/1
    inputs = [
        age,
        1 if transplant == "Yes" else 0,
        1 if chronic == "Yes" else 0,
        1 if cancer == "Yes" else 0,
        surgeries,
        bmi,
        1 if bp_problems == "Yes" else 0,
        1 if diabetes == "Yes" else 0,
        1 if allergies == "Yes" else 0
    ]

    # Convert to array and scale

    feature_cols = ['Age', 'AnyTransplants', 'AnyChronicDiseases',
                'HistoryOfCancerInFamily', 'NumberOfMajorSurgeries',
                'BMI', 'BloodPressureProblems', 'Diabetes', 'KnownAllergies']

    features = pd.DataFrame([inputs], columns=feature_cols)
    scaled_features = scaler.transform(features)


    prediction = model.predict(scaled_features)

    st.divider()
    st.success(f"Estimated Annual Premium: ₹{prediction[0]:,.2f}")

    # This section confirms the logic for your review
    with st.expander("Diagnostic Data"):
        st.write("Feature Order: Age, Trans, Chron, Canc, Surg, BMI, BP, Diab, Allrg")
        st.write("Raw Vector:", inputs)

Overwriting app.py


In [ ]:
from pyngrok import ngrok

# Terminate open tunnels
ngrok.kill()

# Open a tunnel on port 8501
public_url = ngrok.connect(8501, proto="http")
print(f"Click here to open your app: {public_url}")

# Run the app
!streamlit run app.py --server.port 8501

Click here to open your app: NgrokTunnel: "https://detract-halogen-smashing.ngrok-free.dev" -> "http://localhost:8501"





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.196.8.150:8501

  Stopping...
  Stopping...
